In [1]:
!pip -q install chromadb openai langchain tiktoken

In [2]:
!pip show chromadb

In [3]:
## https://www.dropbox.com/s/vs6ocyvpzzncvwh/new_articles.zip

In [4]:
# to download the zip files to local
!wget -q https://www.dropbox.com/s/vs6ocyvpzzncvwh/new_articles.zip

In [5]:
# unzip the zipped files and copy to a folder
!unzip -q new_articles.zip -d new_articles

In [6]:
from langchain.vectorstores import chromadb
from langchain.embeddings import OpenAIEmbeddings
from langchain.llms import OpenAI
from langchain.document_loaders import DirectoryLoader, TextLoader

In [7]:
# for loading data
loader = DirectoryLoader("/new_articles",glob = "./*.txt", loader_cls=TextLoader)

In [8]:
document = loader.load()

In [9]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
text= text_splitter.split_documents(document)

In [10]:
text[0].page_content

In [11]:
len(text)

In [12]:
from langchain import embeddings

In [13]:
persist_directory = 'db'

In [14]:
embedding = OpenAIEmbeddings()

In [15]:
vectordb = Chroma.from_documents(documents=text,
                                embedding=embedding,
                                persist_directory=persist_directory)

In [16]:
#persist db to the disk
vectordb.persist()

In [17]:
vectordb = None

In [18]:
# now we can load the persisted database from disk, and use it as normal db
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)

In [19]:
vectordb

In [20]:
# Make a retriever
retriever = vectordb.as_retriever()

In [21]:
docs = retriever.get_relevant_documents("How much did microsoft raise?")

In [22]:
docs ## you will get mulitple relevant docs from the search operation

In [23]:
retriever = vectordb.as_retriever(search_kwargs={'k':2}) # this will fetch only top 2 similar docs

In [24]:
docs = retriever.get_relevant_documents("How much did microsoft raise?")

In [25]:
len(docs) # this will be 2

In [26]:
## Make a chain
from langchain.chains import RetrievalQA

In [27]:
llm=OpenAI()

In [28]:
llm

In [29]:
qa_chain = RetrievalQA.from_chain_type(llm=llm,
                                      chain_type="stuff",
                                      retriever=retriever,
                                      return_source_documents=True)

In [30]:
def process_llm_response(llm_response):
    print(llm_response['result'])
    print("\n\n Sources:")
    
    for source in llm_response["source_documents"]:
        print(source.metadata['source'])

In [31]:
## full example
query = "How much money did Microsoft raise?"

In [32]:
llm_response = qa_chain(query)

In [33]:
process_llm_response(llm_response)

In [34]:
## delete db

In [35]:
!zip -r db.zip ./db

In [36]:
## to clean up, you can delete the collection
vectordb.delete_collection()
vectordb.persist()

In [37]:
## delete the directory
!rm -rf db/